# 📄 Smart Document Analyst — CNN Training on Colab
**Team:** Benmouma Salma, Gassi Oumaima
**UIR S8** — AI & Big Data 2025–2026 | Prof. Hakim Hafidi

This notebook:
1. Clones the project repo
2. Downloads a subset of the RVL-CDIP dataset
3. Trains a ResNet-18 CNN for document classification
4. Evaluates with confusion matrix + classification report
5. Pushes the trained model back to GitHub

## 1️⃣ Setup — Clone Repo & Install Dependencies
⚠️ **IMPORTANT:** Run the install cell below, then **restart the runtime** (Runtime → Restart session), then continue from cell 1.2.

In [ ]:
# 1.1 — Install dependencies (RUN THIS FIRST, then restart runtime)
!git clone https://github.com/OumaimaGassi/smart-document-analyst.git
%cd smart-document-analyst

# Pin numpy<2.0 FIRST to avoid compatibility issues with sklearn/scipy
!pip install -q "numpy<2.0.0"
!pip install -q crewai crewai-tools google-generativeai pymupdf pytesseract fpdf2 python-dotenv tqdm seaborn datasets

print('\n' + '='*60)
print('⚠️  NOW RESTART THE RUNTIME: Runtime → Restart session')
print('   Then skip this cell and run from cell 1.2 below.')
print('='*60)

In [ ]:
# 1.2 — After restart, run this cell to verify setup
%cd /content/smart-document-analyst

import numpy as np
import torch
print(f'NumPy:   {np.__version__} (should be 1.x)')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA:    {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:     {torch.cuda.get_device_name(0)}')

# Quick sanity check — these should not error
from sklearn.metrics import accuracy_score
from scipy import stats
print('\n✅ NumPy, sklearn, scipy all compatible!')

## 2️⃣ Download RVL-CDIP Dataset Subset

In [ ]:
import os
from datasets import load_dataset
from PIL import Image
from tqdm import tqdm

ALL_CLASSES = [
    'letter', 'memo', 'email', 'file_folder', 'form', 'handwritten',
    'invoice', 'advertisement', 'budget', 'news_article', 'presentation',
    'scientific_publication', 'questionnaire', 'resume', 'scientific_report', 'specification'
]

SUBSET_CLASSES = ['letter', 'form', 'invoice', 'resume', 'scientific_publication', 'email', 'memo', 'advertisement']
SUBSET_INDICES = [ALL_CLASSES.index(c) for c in SUBSET_CLASSES]
IMAGES_PER_CLASS = 2000

print(f'Target classes: {SUBSET_CLASSES}')
print(f'Images per class: {IMAGES_PER_CLASS}')

In [ ]:
# Download using the correct HuggingFace path
print('Loading RVL-CDIP dataset (this may take a few minutes)...')
dataset = load_dataset('aharley/rvl_cdip', split='train', streaming=True)

base_dir = 'data/dataset'
for cls_name in SUBSET_CLASSES:
    os.makedirs(f'{base_dir}/{cls_name}', exist_ok=True)

class_counts = {c: 0 for c in SUBSET_CLASSES}
total_saved = 0
target_total = IMAGES_PER_CLASS * len(SUBSET_CLASSES)

print(f'Downloading {target_total} images...')
for sample in tqdm(dataset, total=target_total * 3, desc='Scanning'):
    label_idx = sample['label']
    if label_idx not in SUBSET_INDICES:
        continue
    cls_name = ALL_CLASSES[label_idx]
    if class_counts[cls_name] >= IMAGES_PER_CLASS:
        continue
    img = sample['image']
    if img.mode != 'RGB':
        img = img.convert('RGB')
    save_path = f'{base_dir}/{cls_name}/{cls_name}_{class_counts[cls_name]:04d}.png'
    img.save(save_path)
    class_counts[cls_name] += 1
    total_saved += 1
    if all(c >= IMAGES_PER_CLASS for c in class_counts.values()):
        break

print(f'\n✅ Downloaded {total_saved} images')
for cls, count in class_counts.items():
    print(f'   {cls}: {count}')

## 3️⃣ Train the CNN Model

In [ ]:
import sys, time, random, json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from tqdm import tqdm
from datetime import datetime

sys.path.insert(0, '.')
from src.models.document_classifier import DocumentClassifierCNN
from src.utils.preprocessing import DOCUMENT_CLASSES_SUBSET, CNN_INPUT_SIZE

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
train_tf = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomRotation(5),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.95, 1.05)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

full_dataset = datasets.ImageFolder(root='data/dataset', transform=train_tf)
print(f'Total images: {len(full_dataset)}')
print(f'Classes: {full_dataset.classes}')

n = len(full_dataset)
n_train, n_val = int(0.8*n), int(0.1*n)
n_test = n - n_train - n_val
train_set, val_set, test_set = random_split(full_dataset, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(SEED))
print(f'Train: {len(train_set)} | Val: {len(val_set)} | Test: {len(test_set)}')

train_loader = DataLoader(train_set, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_set, batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(test_set, batch_size=32, shuffle=False, num_workers=2)

In [ ]:
NUM_CLASSES = len(DOCUMENT_CLASSES_SUBSET)
model = DocumentClassifierCNN(num_classes=NUM_CLASSES, pretrained=True, dropout_rate=0.3).to(device)
print(f'Trainable params: {model.get_trainable_params():,}')

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20, eta_min=1e-6)

In [ ]:
NUM_EPOCHS, PATIENCE = 20, 5
best_val_acc, patience_counter = 0.0, 0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(NUM_EPOCHS):
    model.train()
    tl, tc, tt = 0, 0, 0
    for x, y in tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS}', leave=False):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x); loss = criterion(out, y)
        loss.backward(); optimizer.step()
        tl += loss.item()*x.size(0); _, p = out.max(1)
        tt += y.size(0); tc += p.eq(y).sum().item()
    model.eval()
    vl, vc, vt = 0, 0, 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            out = model(x); loss = criterion(out, y)
            vl += loss.item()*x.size(0); _, p = out.max(1)
            vt += y.size(0); vc += p.eq(y).sum().item()
    ta, va = tc/tt, vc/vt
    history['train_loss'].append(tl/tt); history['train_acc'].append(ta)
    history['val_loss'].append(vl/vt); history['val_acc'].append(va)
    scheduler.step()
    print(f'Epoch {epoch+1:2d}/{NUM_EPOCHS} | Train: {tl/tt:.4f} / {ta:.4f} | Val: {vl/vt:.4f} / {va:.4f}')
    if va > best_val_acc:
        best_val_acc = va; patience_counter = 0
        model.save_model('model/document_classifier.pt')
        print(f'  💾 Saved (val_acc: {va:.4f})')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'Early stopping at epoch {epoch+1}'); break
print(f'\n✅ Best val accuracy: {best_val_acc:.4f}')

## 4️⃣ Evaluation

In [ ]:
best_model = DocumentClassifierCNN.load_model('model/document_classifier.pt', NUM_CLASSES, str(device))
all_preds, all_labels = [], []
with torch.no_grad():
    for x, y in tqdm(test_loader, desc='Testing'):
        out = best_model(x.to(device))
        _, p = out.max(1)
        all_preds.extend(p.cpu().numpy()); all_labels.extend(y.numpy())
print('\n📊 CLASSIFICATION REPORT')
print(classification_report(all_labels, all_preds, target_names=DOCUMENT_CLASSES_SUBSET))
print(f'Accuracy: {accuracy_score(all_labels, all_preds):.4f}')

In [ ]:
# Confusion Matrix + Training Curves
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=DOCUMENT_CLASSES_SUBSET, yticklabels=DOCUMENT_CLASSES_SUBSET)
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True'); axes[0].set_title('Confusion Matrix')
ep = range(1, len(history['train_loss'])+1)
axes[1].plot(ep, history['train_loss'], 'b-o', ms=3, label='Train')
axes[1].plot(ep, history['val_loss'], 'r-o', ms=3, label='Val')
axes[1].set_title('Loss'); axes[1].legend(); axes[1].grid(alpha=0.3)
axes[2].plot(ep, history['train_acc'], 'b-o', ms=3, label='Train')
axes[2].plot(ep, history['val_acc'], 'r-o', ms=3, label='Val')
axes[2].set_title('Accuracy'); axes[2].legend(); axes[2].grid(alpha=0.3)
plt.suptitle('Document Classifier — Benmouma Salma & Gassi Oumaima', fontsize=14)
plt.tight_layout()
os.makedirs('outputs', exist_ok=True)
plt.savefig('outputs/evaluation_results.png', dpi=150)
plt.show()

In [ ]:
meta = {'timestamp': datetime.now().isoformat(), 'team': 'Benmouma Salma, Gassi Oumaima',
    'accuracy': float(accuracy_score(all_labels, all_preds)), 'best_val_acc': float(best_val_acc),
    'num_classes': NUM_CLASSES, 'classes': DOCUMENT_CLASSES_SUBSET,
    'epochs_trained': len(history['train_loss']), 'device': str(device), 'pytorch': torch.__version__}
with open('model/training_metadata.json', 'w') as f:
    json.dump(meta, f, indent=2)
print('✅ Metadata saved')

## 5️⃣ Push to GitHub

In [ ]:
from getpass import getpass
token = getpass('GitHub Personal Access Token: ')
!git remote set-url origin https://{token}@github.com/OumaimaGassi/smart-document-analyst.git
!git config user.name "OumaimaGassi"
!git config user.email "oumaima.gassi@uir.ac.ma"
!git add model/document_classifier.pt model/training_metadata.json outputs/
!git commit -m "feat: add trained CNN model (ResNet-18, RVL-CDIP)"
!git push origin main
print('\n✅ Model pushed to GitHub!')

## ✅ Done!
```bash
git clone https://github.com/OumaimaGassi/smart-document-analyst.git
cd smart-document-analyst && pip install -r requirements.txt
cp .env.example .env  # add GEMINI_API_KEY
python src/main.py --input path/to/document.pdf --hitl
```